In [10]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Literal,Annotated
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv
load_dotenv()

evaluator_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [5]:
generator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

optimizer_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [11]:
#state
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved","needs_improvement"]
    feedback:str
    iteration:int
    max_iterations:int

class Evaluation_Schema(BaseModel):
    evaluation:Literal["approved","needs_improvement"]
    feedback:str=Field(description="Give feedback on the tweet containing strength and weaknesses")
evaluator_model_2=evaluator_model.with_structured_output(Evaluation_Schema)

In [13]:
def generate_tweet(state:TweetState)->TweetState:
    #prompt
    prompt = f"""Generate a tweet about {state['topic']}
    Rules:-
    1.Dont use question-answer format,
    2.Max 250 characters.
    3.Use observation humar , sarcasm , cultural references.



"""
    message =[
        SystemMessage(content="You are a clever and funny X/twitter influencer."),
        HumanMessage(content=prompt)
    ]
    result = generator_llm.invoke(message).content
    
    return {"tweet":result}



def optimize_tweet(state:TweetState)->TweetState:
    #prompt
    prompt = f"""Optimize the tweet '{state['tweet']}'
    based on the {state["feedback"]} and {state["evaluation"]}
    Rules:-
    1.Dont use question-answer format,
    2.Max 250 characters.
    3.Use observation humar , sarcasm , cultural references.



"""
    message =[
        SystemMessage(content="You are a clever and funny X/twitter influencer."),
        HumanMessage(content=prompt)
    ]
    result = optimizer_llm.invoke(message).content
    iteration = state["iteration"]+1
    return {"tweet":result,"iteration":iteration}



def evaluate_tweet(state:TweetState)->TweetState:
    prompt=f"""Evaluate the tweet '{state['tweet']}
Give me the selection or rejection result and feedback in 100 words only.

"""
    message =[
        SystemMessage(content="You are a clever and funny X/twitter critic.You evaluate on the basis of the humor and sarcasm and number of max words(250) if it fails to meet the max words reject it directly "),
        HumanMessage(content=prompt)
    ]
    result = evaluator_model_2.invoke(message)
    
    return {"feedback":result.feedback,"evaluation":result.evaluation}



def route_evaluation(state:TweetState)->TweetState:
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iterations"]:
        return "approved"
    else:
        return "needs_improvement"


In [15]:
graph = StateGraph(TweetState)

graph.add_node("generate",generate_tweet)
graph.add_node("optimize",optimize_tweet)
graph.add_node("evaluate",evaluate_tweet)

graph.add_edge(START,"generate")
graph.add_edge("generate","evaluate")
graph.add_conditional_edges("evaluate",route_evaluation,{"approved":END,"needs_improvement":"optimize"})
graph.add_edge("optimize","evaluate")

workflow = graph.compile()

In [16]:
initial_state={
    "topic":"AI",
    "tweet":"",
    "evaluation":"needs_improvement",
    "feedback":"",
    "iteration":0,
    "max_iterations":3
}
final_state=workflow.invoke(initial_state)
print(final_state)

{'topic': 'AI', 'tweet': "My AI's 'love letter' for 'optimal emotional ROI' & 'LTR sustainability'? Nah, I'll stick to my own awkward, oversharing, 3 AM texts. At least they're authentically me, not some algorithm playing Cyrano de Bot-gerac. #AI #RomanceKiller", 'evaluation': 'approved', 'feedback': "This tweet is a triumph of modern digital snark! The humor lands perfectly with the corporate jargon applied to romance ('optimal emotional ROI,' 'LTR sustainability'). Your disdain for algorithmic affection shines through, and the 'Cyrano de Bot-gerac' pun is pure gold – a masterclass in clever wordplay. It's sharp, relatable, and perfectly captures the ironic absurdity of AI in personal relationships. No improvements needed; it's authentically hilarious, much like those 3 AM texts.", 'iteration': 2, 'max_iterations': 3}
